In [1]:
# bagging and boosting
'''
problem statment: Sets the scene: a payments company wants to catch fraud without
raising too many false alarms on genuine customers. Fraud is rare, which makes it tricky.
'''

In [3]:
# installting libraries:
!pip install scikit-learn matplotlib
# scikit-learn -> functions
# matplotlib -> plot the diff..

In [ ]:
# Step 1 — Create the data
'''
1. What: generate 20,000 example transactions
2. each labelled fraud (1) or genuine (0), with only ~1.5% fraud.
'''

# step2 - creating frames
'''
1. keep 70% for training,
2. 30% for testing, using stratify to preserve the fraud ratio in both halves.
'''

# step 3 - The three models
'''
1. Single tree (before) — one decision tree, our baseline. [ cluster of data]
2. Bagging (after) — 200 trees averaged; very stable, cuts variance. (bagging -> var)
3. Boosting (after) — 200 trees in a chain, each fixing the previous one's mistakes; (boosting -> BIAS)
cuts bias, focuses on the hard fraud cases.
'''

# step 4 : the Train each model and measure it
'''
1. for each model: train it, predict on the test set, and measure the result.
'''

# step 5 : compare side by side ( before bagging / bosting -> after apply)
'''
1. metrices / rubics comparison
'''

In [4]:
'''
importing all necessary libaries
'''

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)


In [5]:

RANDOM_STATE = 42


def banner(title):
    print("\n" + "=" * 64)
    print(title)
    print("=" * 64)

In [6]:
banner("STEP 0  —  THE PROBLEM")
print(
    "We run a payments company. A few transactions are FRAUD, most are\n"
    "genuine. We want a model that catches fraud WITHOUT annoying real\n"
    "customers with false alarms. Fraud is very rare, so this is tricky."
)


STEP 0  —  THE PROBLEM
We run a payments company. A few transactions are FRAUD, most are
genuine. We want a model that catches fraud WITHOUT annoying real
customers with false alarms. Fraud is very rare, so this is tricky.


In [7]:
banner("STEP 1  —  CREATE THE DATA")
print("WHY: we need example transactions, each labelled fraud (1) or genuine (0).")
X, y = make_classification(

    n_samples=20000,
    n_features=12,
    n_informative=8,
    weights=[0.99, 0.01],
    class_sep=1.3,

    random_state=RANDOM_STATE,
)


STEP 1  —  CREATE THE DATA
WHY: we need example transactions, each labelled fraud (1) or genuine (0).


In [8]:
n_fraud = int(y.sum())
print(f"\nTotal transactions : {len(y):,}")
print(f"Genuine            : {len(y) - n_fraud:,}")
print(f"Fraud              : {n_fraud:,}  (only {y.mean()*100:.2f}% of all data!)")
print("\nNOTE: because fraud is so rare, 'accuracy' is a trap — a lazy model that")
print("calls everything 'genuine' would be ~98% accurate but catch ZERO fraud.")


Total transactions : 20,000
Genuine            : 19,695
Fraud              : 305  (only 1.52% of all data!)

NOTE: because fraud is so rare, 'accuracy' is a trap — a lazy model that
calls everything 'genuine' would be ~98% accurate but catch ZERO fraud.


In [9]:
banner("STEP 2  —  SPLIT INTO TRAIN & TEST")
print("WHY: we train on one part, then test on data the model has never seen,")
print("to measure REAL performance (not memorisation). 'stratify' keeps the")
print("same fraud ratio in both halves so the test is fair.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
print(f"\nTraining transactions : {len(y_train):,}")
print(f"Testing transactions  : {len(y_test):,}  (fraud in test: {int(y_test.sum())})")



STEP 2  —  SPLIT INTO TRAIN & TEST
WHY: we train on one part, then test on data the model has never seen,
to measure REAL performance (not memorisation). 'stratify' keeps the
same fraud ratio in both halves so the test is fair.

Training transactions : 14,000
Testing transactions  : 6,000  (fraud in test: 91)


In [10]:
banner("STEP 3  —  THE THREE MODELS WE COMPARE")
print("BEFORE -> Single tree : one model, our baseline.")
print("AFTER  -> Bagging     : 200 trees averaged  (cuts variance, very stable).")
print("AFTER  -> Boosting    : 200 trees in a chain, each fixing the last's")
print("                        mistakes (cuts bias, focuses on hard fraud).")
models = {
    "Single tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Bagging":     RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                          random_state=RANDOM_STATE, n_jobs=-1),
    "Boosting":    GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

# single model -> single tree -> will not be solving the problem..
# bagging / boosting -> techniques -> functions
# bagging -> random forest
# boosting -> gradientBoosting

# bagging -> Variance -> Classifier -> Majority (voting)
# boosting -> BIAS -> Chainning -> AVG (error fixing)


STEP 3  —  THE THREE MODELS WE COMPARE
BEFORE -> Single tree : one model, our baseline.
AFTER  -> Bagging     : 200 trees averaged  (cuts variance, very stable).
AFTER  -> Boosting    : 200 trees in a chain, each fixing the last's
                        mistakes (cuts bias, focuses on hard fraud).


In [12]:
banner("STEP 4  —  TRAIN EACH MODEL & MEASURE IT")
print("For each model we: (a) train it, (b) predict on the test set,")
print("(c) measure how well it found fraud.\n")

results = {}
for name, model in models.items():
    print("-" * 64)
    print(f">> {name}: training on {len(y_train):,} transactions...")
    model.fit(X_train, y_train)
    print("   training done. Now predicting on unseen test data...")

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)
    results[name] = {"precision": precision, "recall": recall, "f1": f1, "auc": auc}

    print(f"   Caught {tp} of {tp+fn} real frauds, with {fp} false alarms.")
    print(f"   Precision = {precision:.2f}  (of flagged, how many were truly fraud)")
    print(f"   Recall    = {recall:.2f}  (of all fraud, how much we caught)")
    print(f"   F1        = {f1:.2f}  (balance of precision & recall)")
    print(f"   ROC-AUC   = {auc:.2f}  (how well it ranks fraud above genuine)")



STEP 4  —  TRAIN EACH MODEL & MEASURE IT
For each model we: (a) train it, (b) predict on the test set,
(c) measure how well it found fraud.

----------------------------------------------------------------
>> Single tree: training on 14,000 transactions...
   training done. Now predicting on unseen test data...
   Caught 27 of 91 real frauds, with 70 false alarms.
   Precision = 0.28  (of flagged, how many were truly fraud)
   Recall    = 0.30  (of all fraud, how much we caught)
   F1        = 0.29  (balance of precision & recall)
   ROC-AUC   = 0.64  (how well it ranks fraud above genuine)
----------------------------------------------------------------
>> Bagging: training on 14,000 transactions...
   training done. Now predicting on unseen test data...
   Caught 19 of 91 real frauds, with 0 false alarms.
   Precision = 1.00  (of flagged, how many were truly fraud)
   Recall    = 0.21  (of all fraud, how much we caught)
   F1        = 0.35  (balance of precision & recall)
   ROC-AUC

In [13]:
banner("STEP 5  —  COMPARE THEM SIDE BY SIDE")
print(f"{'Model':14}{'Precision':>11}{'Recall':>9}{'F1':>7}{'ROC-AUC':>9}")
print("-" * 50)
for name, m in results.items():
    print(f"{name:14}{m['precision']:>11.2f}{m['recall']:>9.2f}{m['f1']:>7.2f}{m['auc']:>9.2f}")
print("-" * 50)
print("Takeaway: both ensembles beat the single tree. Bagging is the most")
print("PRECISE (fewest false alarms); Boosting has the best overall ranking")
print("(highest ROC-AUC). The single tree is the noisiest, least reliable one.")

banner("STEP 6  —  SHOW THE CHART (saved as comparison.png)")
metrics = ["precision", "recall", "f1", "auc"]
nice = ["Precision", "Recall", "F1", "ROC-AUC"]
labels = list(results.keys())
colors = ["#888780", "#1D9E75", "#7F77DD"]
x = np.arange(len(metrics))
width = 0.25
plt.figure(figsize=(9, 5))
for i, name in enumerate(labels):
    plt.bar(x + i * width, [results[name][k] for k in metrics], width,
            label=name, color=colors[i])
plt.xticks(x + width, nice)
plt.ylabel("Score (higher is better)")
plt.title("Single model vs Bagging vs Boosting")
plt.legend()
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig("comparison.png", dpi=130)
print("Chart saved. Open comparison.png to see the bars side by side.")



STEP 5  —  COMPARE THEM SIDE BY SIDE
Model           Precision   Recall     F1  ROC-AUC
--------------------------------------------------
Single tree          0.28     0.30   0.29     0.64
Bagging              1.00     0.21   0.35     0.77
Boosting             0.42     0.29   0.34     0.79
--------------------------------------------------
Takeaway: both ensembles beat the single tree. Bagging is the most
PRECISE (fewest false alarms); Boosting has the best overall ranking
(highest ROC-AUC). The single tree is the noisiest, least reliable one.

STEP 6  —  SHOW THE CHART (saved as comparison.png)
Chart saved. Open comparison.png to see the bars side by side.
